<a href="https://colab.research.google.com/github/Viswas-Vinayakumar/AI-search/blob/main/mini_search_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini search: BM25 + vectors + RRF + metrics

5 synthetic coaches, 3 test queries. Run the cells top to bottom (no GPU needed).

In [9]:
!python -m spacy download de_core_news_md -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 MB 14.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [10]:
import math, re
import numpy as np
import spacy

## **1. Data and answer key**

In [3]:
COACHES = {
    "c1": "Konfliktcoach für Teams: Spannungen lösen, Mediation und schwierige Gespräche.",
    "c2": "Scrum-Trainerin: Einführung von Scrum und Kanban in Softwareteams.",
    "c3": "Vertriebstraining: Kaltakquise und Abschlusstechniken für Verkäufer.",
    "c4": "Resilienz und Stressmanagement: Burnout vorbeugen und mit Druck umgehen.",
    "c5": "Mediation bei Streit zwischen Abteilungen, neutral und vertraulich.",
}
GOLDEN = {
    "Scrum Einführung":          {"c2"},
    "Streit im Team schlichten": {"c1", "c5"},
    "Stress bei der Arbeit":     {"c4"},
}
print("coaches:", len(COACHES), "| test queries:", len(GOLDEN))

coaches: 5 | test queries: 3


## **2. Tokenizer**

In [4]:
STOP = {"für", "und", "von", "in", "bei", "mit", "der", "die", "das", "im", "zwischen"}
def tokenize(text):
    words = re.findall(r"\w+", text.lower())
    return [w for w in words if w not in STOP]
print("\n" + str(tokenize(COACHES["c2"])))


['scrum', 'trainerin', 'einführung', 'scrum', 'kanban', 'softwareteams']


## **3. BM25 (keyword search), from scratch**

In [5]:
docs = {cid: tokenize(t) for cid, t in COACHES.items()}
N = len(docs)
avg_len = sum(len(d) for d in docs.values()) / N
k1, b = 1.5, 0.75

def idf(word):
    n = sum(word in d for d in docs.values())
    return math.log(1 + (N - n + 0.5) / (n + 0.5))

def bm25_score(query, doc):
    score = 0.0
    for w in tokenize(query):
        tf = doc.count(w)
        if tf:
            score += idf(w) * tf * (k1 + 1) / (tf + k1 * (1 - b + b * len(doc) / avg_len))
    return score

def keyword_rank(query):
    scores = {cid: bm25_score(query, d) for cid, d in docs.items()}
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    return [(cid, s) for cid, s in ranked if s > 0]

print(f"\naverage length: {avg_len:.1f} words")
print("IDF('scrum') =", round(idf("scrum"), 3), "| IDF('mediation') =", round(idf("mediation"), 3))
for q in GOLDEN:
    print(f"{q:<28} ->", [(c, round(s, 3)) for c, s in keyword_rank(q)] or "nothing found")


average length: 5.6 words
IDF('scrum') = 1.386 | IDF('mediation') = 0.875
Scrum Einführung             -> [('c2', 3.279)]
Streit im Team schlichten    -> [('c5', 1.457)]
Stress bei der Arbeit        -> nothing found


## **4. Vectors: averaged word vectors + cosine**

In [6]:
nlp = spacy.load("de_core_news_md")

def embed(text):
    vecs = [t.vector for t in nlp(text) if t.is_alpha and not t.is_stop and t.has_vector]
    v = np.mean(vecs, axis=0)                 # mean pooling
    return v / np.linalg.norm(v)              # normalise to length 1

coach_vecs = {cid: embed(t) for cid, t in COACHES.items()}
print("\neach text becomes", len(coach_vecs["c1"]), "numbers; first 4 of c1:", np.round(coach_vecs["c1"][:4], 3))

def vector_rank(query):
    q = embed(query)
    scores = {cid: float(v @ q) for cid, v in coach_vecs.items()}   # cosine similarity
    return sorted(scores.items(), key=lambda x: -x[1])

for q in GOLDEN:
    print(f"{q:<28} ->", [(c, round(s, 3)) for c, s in vector_rank(q)])


each text becomes 300 numbers; first 4 of c1: [ 0.074  0.035  0.044 -0.121]
Scrum Einführung             -> [('c2', 0.797), ('c1', 0.532), ('c3', 0.504), ('c4', 0.481), ('c5', 0.463)]
Streit im Team schlichten    -> [('c1', 0.62), ('c5', 0.584), ('c4', 0.432), ('c3', 0.357), ('c2', 0.253)]
Stress bei der Arbeit        -> [('c4', 0.685), ('c1', 0.556), ('c5', 0.488), ('c3', 0.32), ('c2', 0.254)]


## **5. Reciprocal Rank Fusion (RRF)**

In [7]:
def hybrid_rank(query, k=60):
    fused = {}
    for ranking in (keyword_rank(query), vector_rank(query)):
        for rank, (cid, _) in enumerate(ranking, start=1):
            fused[cid] = fused.get(cid, 0.0) + 1 / (k + rank)
    return sorted(fused.items(), key=lambda x: -x[1])

q = "Streit im Team schlichten"
kw = {c: r for r, (c, _) in enumerate(keyword_rank(q), 1)}
vc = {c: r for r, (c, _) in enumerate(vector_rank(q), 1)}
print(f"\nQUERY: {q}")
print("coach | keyword rank | vector rank | RRF")
for cid, s in hybrid_rank(q):
    print(f"{cid:<5} | {str(kw.get(cid, '-')):<12} | {vc.get(cid, '-'):<11} | {s:.4f}")


QUERY: Streit im Team schlichten
coach | keyword rank | vector rank | RRF
c5    | 1            | 2           | 0.0325
c1    | -            | 1           | 0.0164
c4    | -            | 3           | 0.0159
c3    | -            | 4           | 0.0156
c2    | -            | 5           | 0.0154


## **6. Metrics: RR, MRR, Recall@3**

In [8]:
METHODS = {"keyword": keyword_rank, "vector": vector_rank, "hybrid": hybrid_rank}
def recall_at(ranked, relevant, k=3):
    return len(set(ranked[:k]) & relevant) / len(relevant)
def reciprocal_rank(ranked, relevant):
    for pos, cid in enumerate(ranked, start=1):
        if cid in relevant:
            return 1 / pos
    return 0.0

print()
for name, fn in METHODS.items():
    rr_list, rec_list = [], []
    for q, rel in GOLDEN.items():
        ranked = [c for c, _ in fn(q)]
        rr_list.append(reciprocal_rank(ranked, rel))
        rec_list.append(recall_at(ranked, rel))
    print(f"{name:<8} RR per query {[round(x, 2) for x in rr_list]}  ->  MRR {np.mean(rr_list):.2f} | Recall@3 {np.mean(rec_list):.2f}")


keyword  RR per query [1.0, 1.0, 0.0]  ->  MRR 0.67 | Recall@3 0.50
vector   RR per query [1.0, 1.0, 1.0]  ->  MRR 1.00 | Recall@3 1.00
hybrid   RR per query [1.0, 1.0, 1.0]  ->  MRR 1.00 | Recall@3 1.00
